# NB3 · Non-linearita', ensemble e il caso retail Conad

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB3_ensemble_caso_retail.ipynb)

**albero, random forest, boosting e SVM: dall'esempio ISL al dataset retail**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**; esegui le celle in ordine, dall'alto verso il basso.

doppio binario: prima vediamo la **meccanica** dei metodi ad albero su un dataset di ISL (*Carseats*, capitolo 8), poi la portiamo sul **caso retail Conad** (dataset sintetico). chiudiamo con un cenno alle SVM (capitolo 9).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def carica(nome):
    "carica un csv dal repo GitHub (Colab) o dalla cartella locale ../data (fallback)"
    base = "https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/"
    try:
        return pd.read_csv(base + nome)
    except Exception:
        return pd.read_csv("../data/" + nome)

## 1. l'albero di decisione (esempio ISL, Carseats)

dataset **Carseats** di ISL: vendite di seggiolini auto in 400 negozi. costruiamo la variabile `High` (vendite oltre le 8.000 unita') e addestriamo un alberello: si legge come una sequenza di domande si/no. e' la figura 8.1 del libro.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

cs = carica("Carseats.csv")
cs["High"] = (cs["Sales"] > 8).astype(int)
Xcs = pd.get_dummies(cs.drop(columns=["Sales", "High"]),
                     columns=["ShelveLoc", "Urban", "US"], drop_first=True)
alberello = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xcs, cs["High"])

plt.figure(figsize=(15, 7))
plot_tree(alberello, feature_names=Xcs.columns, class_names=["normali", "High"],
          filled=True, fontsize=9, impurity=False)
plt.title("Albero di decisione su Carseats  (cfr. ISL fig. 8.1)")
plt.show()

l'albero e' trasparente: ogni nodo e' una domanda su una variabile. ma un solo albero, lasciato crescere, e' instabile e tende a overfittare. ora portiamo questi metodi sul caso retail.

## 2. il dataset retail Conad (sintetico)

10.000 clienti di una carta fedelta'. due compiti sugli **stessi dati**:

- **churn** (classificazione): il cliente e' a rischio abbandono? (0/1);
- **spesa prevista a 12 mesi** (regressione): quanto spendera'? (euro).

In [ ]:
df = carica("conad_retail.csv")
print("righe:", len(df), " colonne:", df.shape[1])

cat = ["formato_pdv", "canale", "reparto_preferito"]
X = pd.get_dummies(df.drop(columns=["cliente_id", "churn", "spesa_prevista_12m"]), columns=cat)
y_churn = df["churn"]
y_spesa = df["spesa_prevista_12m"]
print("numero di feature dopo la codifica:", X.shape[1])
df.head()

## 3. churn: un albero che overfitta

l'albero e' intuitivo, ma se lo lasciamo crescere troppo impara il rumore: ottimo sul training, peggiore sul test.

> **manopola**: cambia `PROFONDITA` (prova 2, 4, 8, 20) e confronta AUC di training e di test. da un certo punto in poi il training migliora ma il test peggiora: e' overfitting.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

Xtr, Xte, ytr, yte = train_test_split(X, y_churn, test_size=0.30, random_state=0, stratify=y_churn)

# >>> MANOPOLA: profondita' massima dell'albero <<<
PROFONDITA = 8

albero = DecisionTreeClassifier(max_depth=PROFONDITA, random_state=0).fit(Xtr, ytr)
auc_tr = roc_auc_score(ytr, albero.predict_proba(Xtr)[:, 1])
auc_te = roc_auc_score(yte, albero.predict_proba(Xte)[:, 1])
print(f"profondita' {PROFONDITA}  ->  AUC training: {auc_tr:.3f}   AUC test: {auc_te:.3f}")
print("differenza training - test (piu' e' grande, piu' overfitta):", round(auc_tr - auc_te, 3))

## 4. la random forest stabilizza

tanti alberi diversi mediati insieme: meno varianza, di solito molto piu' accurata. e ci dice quali variabili contano davvero.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
print("AUC test random forest:", round(roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]), 3))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(7.5, 4.5))
imp[::-1].plot(kind="barh", color="#00ADCF")
plt.title("Random forest: importanza delle variabili (churn)"); plt.tight_layout(); plt.show()

le variabili in cima (recency, visite, distanza, reclami) sono i veri segnali del churn; in fondo finiscono le feature di rumore (meteo, giorno di iscrizione, sondaggio).

## 5. gradient boosting

alberi piccoli in sequenza, ognuno corregge gli errori del precedente: spesso il piu' accurato.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
print("AUC test gradient boosting:", round(roc_auc_score(yte, gb.predict_proba(Xte)[:, 1]), 3))

## 6. SVM: il confine in 2D

per vedere il **confine di decisione** usiamo solo due variabili (recency e visite al mese) e una SVM con kernel non lineare. il confine separa la zona "a rischio churn" dal resto.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

due = ["recency_giorni", "n_visite_mese"]
X2 = df[due].values
X2tr, X2te, y2tr, y2te = train_test_split(X2, y_churn, test_size=0.30, random_state=0, stratify=y_churn)
svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0)).fit(X2tr, y2tr)

x_min, x_max = df["recency_giorni"].min(), df["recency_giorni"].max()
y_min, y_max = df["n_visite_mese"].min(), df["n_visite_mese"].max()
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7.5, 5))
plt.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
plt.scatter(X2te[:, 0], X2te[:, 1], c=y2te, cmap="coolwarm", s=10, alpha=0.5)
plt.xlim(x_min, x_max); plt.ylim(y_min, y_max)
plt.xlabel("recency (giorni dall'ultimo acquisto)"); plt.ylabel("visite al mese")
plt.title("SVM: confine di decisione per il churn"); plt.show()

## 7. compito B, previsione spesa: lineare vs boosting

stesso dataset, ora il target e' continuo (euro). qui si vede che i modelli flessibili battono il lineare grazie alle non-linearita'.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

Xtr, Xte, str_, ste = train_test_split(X, y_spesa, test_size=0.30, random_state=0)
lin = LinearRegression().fit(Xtr, str_)
gbr = GradientBoostingRegressor(random_state=0).fit(Xtr, str_)
print("R2 test lineare:          ", round(r2_score(ste, lin.predict(Xte)), 3))
print("R2 test gradient boosting:", round(r2_score(ste, gbr.predict(Xte)), 3))

**quale modello per quale obiettivo?** se conta spiegare, parti dal lineare interpretabile; se conta solo prevedere bene, gli ensemble di solito vincono. e' di nuovo il trade-off flessibilita' / interpretabilita' del mattino.

---

### cella bonus: permutation importance

l'importanza basata sull'impurita' favorisce le variabili continue. la *permutation importance* e' piu' onesta: mescola una colonna alla volta e misura quanto peggiora il modello. le feature di rumore vanno vicino a zero.

In [ ]:
# BONUS
from sklearn.inspection import permutation_importance

Xtr, Xte, ytr, yte = train_test_split(X, y_churn, test_size=0.30, random_state=0, stratify=y_churn)
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
pi = permutation_importance(rf, Xte, yte, n_repeats=5, random_state=0, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
imp[::-1].plot(kind="barh", figsize=(7.5, 4.5), color="#002060")
plt.title("Permutation importance (churn)"); plt.tight_layout(); plt.show()